### Load the preprocessed data

In [1]:
import pandas as pd

X_train = pd.read_csv('dataset/X_train.csv')
y_train = pd.read_csv('dataset/y_train.csv').squeeze()
X_test = pd.read_csv('dataset/X_test.csv')
y_test = pd.read_csv('dataset/y_test.csv').squeeze()

X_train_balanced = pd.read_csv('dataset/X_train_balanced.csv')
y_train_balanced = pd.read_csv('dataset/y_train_balanced.csv').squeeze()

print("Raw train:", X_train.shape, "| Balanced train:", X_train_balanced.shape, "| Test:", X_test.shape)

Raw train: (25217, 47) | Balanced train: (39544, 47) | Test: (6305, 47)


### Build a shared results table

In [2]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

def evaluate_and_log(name, model, X_tr, y_tr, X_te, y_te, cv):
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score)

    cv_auc = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    row = {
        'model': name,
        'cv_roc_auc_mean': round(cv_auc.mean(), 3),
        'test_accuracy': round(accuracy_score(y_te, y_pred), 3),
        'test_precision': round(precision_score(y_te, y_pred), 3),
        'test_recall': round(recall_score(y_te, y_pred), 3),
        'test_f1': round(f1_score(y_te, y_pred), 3),
        'test_roc_auc': round(roc_auc_score(y_te, y_proba), 3),
    }
    results.append(row)
    print(row)
    return model

### Train the KNN baseline (raw training data)

In [3]:
from sklearn.neighbors import KNeighborsClassifier

knn_baseline = evaluate_and_log('KNN (baseline, raw)', KNeighborsClassifier(),
                                  X_train, y_train, X_test, y_test, cv)

{'model': 'KNN (baseline, raw)', 'cv_roc_auc_mean': np.float64(0.783), 'test_accuracy': 0.832, 'test_precision': 0.68, 'test_recall': 0.416, 'test_f1': 0.516, 'test_roc_auc': 0.788}


### Train the same model on SMOTE-balanced data, and compare

In [4]:
knn_balanced = evaluate_and_log('KNN (baseline, SMOTE-balanced)', KNeighborsClassifier(),
                                  X_train_balanced, y_train_balanced, X_test, y_test, cv)

{'model': 'KNN (baseline, SMOTE-balanced)', 'cv_roc_auc_mean': np.float64(0.951), 'test_accuracy': 0.751, 'test_precision': 0.448, 'test_recall': 0.67, 'test_f1': 0.537, 'test_roc_auc': 0.778}


### Hyperparameter tuning

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': [5, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 3))

knn_tuned = evaluate_and_log('KNN (tuned)', grid.best_estimator_,
                               X_train, y_train, X_test, y_test, cv)

Best params: {'metric': 'manhattan', 'n_neighbors': 21, 'weights': 'distance'}
Best CV ROC-AUC: 0.839
{'model': 'KNN (tuned)', 'cv_roc_auc_mean': np.float64(0.839), 'test_accuracy': 0.853, 'test_precision': 0.832, 'test_recall': 0.402, 'test_f1': 0.542, 'test_roc_auc': 0.849}


### Compare tuned vs. baseline

In [6]:
pd.DataFrame(results)

,model,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,"KNN (baseline, raw)",0.783,0.832,0.680,0.416,0.516,0.788
1,"KNN (baseline, SMOTE-balanced)",0.951,0.751,0.448,0.670,0.537,0.778
2,KNN (tuned),0.839,0.853,0.832,0.402,0.542,0.849
